In [ ]:
# Set up the file path
import os
os.chdir('..')

In [ ]:
# Import packages
import numpy as np
from itertools import product
from RL4CRN.iocrns.iocrn import IOCRN
from RL4CRN.iocrns.reactions import MassAction
from RL4CRN.utils.ic import IC
from RL4CRN.iocrns.reaction_library import construct_mass_action_library
from RL4CRN.rewards.deterministic import dynamic_tracking_error

In [ ]:
# Create mass-action reactions
r1 = MassAction([], ['Z_1'], ['u_1'], [1.])
r2 = MassAction(['X_2'], ['X_2', 'Z_2'], [None], [2])
r3 = MassAction(['Z_1', 'Z_2'], [], [None], [100])
r4 = MassAction(['Z_1'], ['Z_1', 'X_1'], [None], [3])
r5 = MassAction(['X_1'], ['X_1', 'X_2'], [None], [2])
r6 = MassAction(['X_1'], [], [None], [2])
r7 = MassAction(['X_2'], [], [None], [2])

In [ ]:
# Create and compile the IOCRN
iocrn = IOCRN(reactions=[r1, r2, r3, r4, r5, r6, r7], output_labels=['X_2'])
iocrn.compile()
num_inputs = iocrn.num_inputs
print(iocrn)

In [ ]:
# Construct initial conditions
ic = IC(iocrn.species_labels, values= [
    [0, 0, 0, 0],
])
x0_list = ic.get_ic(iocrn)
print(x0_list)

In [ ]:
# Construct inputs
nums = [0.5, 1.0, 1.5]
u_list = [np.array(u) for u in product(nums, repeat=num_inputs)] # list of input combinations, each input is a numpy array of shape (p,)
print(u_list)

In [ ]:
# Simulate Dynamics
time_horizon = np.linspace(0, 100.0, 1000, dtype=np.float32)
t, x_list, y_list, task_info = iocrn.transient_response(u_list, x0_list, time_horizon)

# Plot using class method
fig, ax = iocrn.plot_transient_response(alpha = 1)

In [ ]:
# Construct the weights for the performance metric
N_t = 1000
w = np.zeros(N_t)
w[(len(w)//5)*4:] = w[(len(w)//5)*4:] + 1
w[:(len(w)//5)] = w[:(len(w)//5)]*0
w = w[np.newaxis, :]

# Compute Reward
def compute_reward(state):
   r_list = [np.array([u[0]]) for u in u_list]
   print(r_list)
   x0_list = ic.get_ic(state)
   out = dynamic_tracking_error(state, u_list, x0_list, time_horizon, r_list, w, norm=1, LARGE_NUMBER=1e4)
   return out

print(compute_reward(iocrn))

In [ ]:
iocrn.reactions[0].set_parameters([10.0])
t, x_list, y_list, task_info = iocrn.transient_response(u_list, x0_list, time_horizon)
fig, ax = iocrn.plot_transient_response(alpha = 1)
